# Phase 6 — Lesion-grounded XAI battery (pre-registered family K)

**Everything frozen; the only backward passes compute activation gradients for
attribution (Grad-CAM). There is no optimizer anywhere in this notebook.**

Preregistered axes (methods §\ref{subsec:phase6_preregistered}):
- **(a) Plausibility** — explanation concentration in the TN5000 expert boxes:
  localisation ratio = in-box share / box-area share (mask-region precision for the
  descriptor heads; CAM energy for ERM), + descriptive pointing game.
- **(b) Faithfulness** — deletion (PRIMARY) / insertion under per-image Gaussian blur
  ($\sigma{=}8$), 8 steps, raw probabilities (raw-raw), masks FIXED at the original
  prediction, one seeded random ordering per image as control; faithfulness gap =
  trapezoid area difference (random − guided for deletion; guided − random for insertion).
- **(c) Stability** — seeded noise ($\sigma{=}0.02$) + affine jitter ($\pm3°/\pm5$px),
  full pipeline recomputed, Spearman correlation of clean-vs-perturbed CAMs.
- **(d) Randomisation sanity** — cascade reinit (R1p head; ERM fc → fc+layer4 →
  +layer3), probe n=160; self-gate: mean |ρ| < 0.2 at final cascade stage (asserted).

**Attribution objects:** ELA (Exact Linear Attribution) for the linear descriptor heads
(R0/R1p/R2p 3c-retrained heads; JOINT-g/JOINT-img saved heads) — exact by construction,
gated at |Σattribution − logit| < 1e-3 on a seeded probe; Grad-CAM (layer4) for ERM, with
Eigen-CAM + channel-subsampled Score-CAM as descriptive robustness rows.
CAM convention everywhere: **ReLU → ℓ1 normalisation**.

**Family K (pre-registered; 5,000 paired bootstrap resamples; two-sided empirical p;
Bonferroni α = 0.05/4 = 0.0125; seed 42):**
K1 localisation ratio R1p − ERM (TN5000-test, PRIMARY plausibility);
K2 deletion gap R1p − ERM (TN5000-test);
K3 deletion gap R1p − ERM (ThyUS2Path seeded panel; label-free ⇒ independent of the
Phase-5 licence);
K4 deletion gap JOINT-g − JOINT-img (TN5000-test; G3 mirror in explanation space).
Everything else is descriptive with pre-declared expectations.

## Attachments (Kaggle datasets to add before Run-All)
- raw: `tn3k-data` (support episode), `tn5000` (images + Annotations XML + ImageSets)
- checkpoints: Phase-1 `best_resnet34_seed42.pt`, Phase-2 `best_fusion_decoder.pt`
- Phase-3 artifacts: `phase3_classification/` (`features_*.npz`,
  `erm_resnet34_tn5000.pt`, `erm_resnet34_probs.json`, `run_summary.json`)
- Phase-3c artifacts: `linear_heads_retrained.json`
- Phase-4 artifacts: `joint_g_trunk.pt`, `joint_img_trunk.pt`,
  `phase4_joint_probs_tn5000.json`
- **Phase-5 outputs**: `phase5_thyus2path/` (`phase5_manifest_unique_cases.csv`,
  `phase5_bank_thyus2path.npz`)
- ThyUS2Path raw dataset (the Phase-5 manifest stores absolute image paths)

In [ ]:
import os, re, json, time, hashlib, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from scipy.stats import spearmanr
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE, MASK_THRESH, PRED_THRESH = 384, 127, 0.5
SEG_K, RING_PX = 20, 15
PROTO_SCALES, SCORE_HW = ["e2", "e3", "e4"], 48
FEAT_SCALES = ["e3", "e4"]
BATCH = 16
ERM_REF_AUROC, R1P_REF_AUROC, GATE_TOL_AUROC = 0.9401, 0.8902, 0.002
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# ---- Phase-6 protocol constants (pre-registered) ----
N_BOOT_K   = 5000
ALPHA_K    = 0.05 / 4
THY_PANEL_N = 600          # ThyUS2Path case-first-image panel (seeded)
DEL_STEPS  = 8             # deletion/insertion steps
BLUR_SIGMA = 8.0           # per-image Gaussian blur baseline
ELA_EXACT_TOL = 1e-3       # exactness gate (asserted)
ELA_PROBE_N = 32           # seeded exactness probe size
STAB_N, RAND_N = 160, 160  # stability / cascade probe sizes
NOISE_SD, AFF_ROT, AFF_SH = 0.02, 3.0, 5.0
RAND_GATE_RHO = 0.2        # cascade sanity gate (asserted)
INTEGRITY_TOL = 5e-5       # live-vs-archive prob agreement (fp32 batch-shape roundoff)
SCORECAM_TOP, SCORECAM_N = 64, 150
MONTAGE_N = 8

OUTPUT_ROOT = Path("/kaggle/working/phase6_xai")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def rglob1(patterns, roots=("/kaggle/input", "/kaggle/working")):
    """First existing match (files or dirs) for any glob; deterministic sort."""
    hits = []
    for pat in patterns:
        for root in roots:
            hits += sorted(Path(root).rglob(pat))
    return [h for h in hits if h.exists()]

ART = {}
def need(key, patterns, kind="file"):
    hits = rglob1(patterns)
    ART[key] = hits[0] if hits else None
    return ART[key]

need("ckpt_phase1", ["best_resnet34_seed42.pt"])
need("ckpt_decoder", ["best_fusion_decoder.pt"])
need("p3_summary",  ["phase3_classification/run_summary.json", "run_summary.json"])
need("lin_retr",    ["phase3c_external/linear_heads_retrained.json", "linear_heads_retrained.json"])
need("erm_ckpt",    ["phase3_classification/erm_resnet34_tn5000.pt", "erm_resnet34_tn5000.pt"])
need("erm_probs",   ["phase3_classification/erm_resnet34_probs.json", "erm_resnet34_probs.json"])
need("tn5k_fval",   ["phase3_classification/features_val.npz", "features_val.npz"])
need("tn5k_ftest",  ["phase3_classification/features_test.npz", "features_test.npz"])
need("joint_g",     ["phase4_joint/joint_g_trunk.pt", "joint_g_trunk.pt"])
need("joint_img",   ["phase4_joint/joint_img_trunk.pt", "joint_img_trunk.pt"])
need("joint_probs", ["phase4_joint/phase4_joint_probs_tn5000.json", "phase4_joint_probs_tn5000.json"])
need("tn3k_root",   ["tn3k-data/Thyroid Dataset/tn3k", "*Thyroid Dataset/tn3k"], kind="dir")
need("tn5000_root", ["tn5000/TN5000_forReview", "*TN5000_forReview"], kind="dir")
need("p5_manifest", ["phase5_thyus2path/phase5_manifest_unique_cases.csv",
                     "phase5_manifest_unique_cases.csv"])
need("p5_bank",     ["phase5_thyus2path/phase5_bank_thyus2path.npz",
                     "phase5_bank_thyus2path.npz"])

REQUIRED = ["ckpt_phase1", "ckpt_decoder", "p3_summary", "lin_retr", "erm_ckpt",
            "erm_probs", "tn5k_fval", "tn5k_ftest", "joint_g", "joint_img",
            "joint_probs", "tn3k_root", "tn5000_root", "p5_manifest", "p5_bank"]
missing = [k for k in REQUIRED if ART.get(k) is None]
assert not missing, f"missing artifacts {missing} - see the attachments markdown cell"
for k in REQUIRED:
    print(f"  {k:12s} -> {ART[k]}")
print("device:", DEVICE, "| output:", OUTPUT_ROOT)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))


## 1. Frozen machinery I — loaders + canonical Phase-3c TN3K support chain
Byte-identical to Phase 5 (itself byte-identical to 3c/3e/4): sha256-dedup → seed-42
15% val carve-out → train pool → seed-52 K=20 support pick, with the hard
episode-identity assert against the Phase-3 archive.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci

def letterbox_mask(msk_raw, size=IMG_SIZE):
    h, w = msk_raw.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    rm = cv2.resize(msk_raw, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    cm = np.zeros((size, size), np.uint8); cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return cm

def make_ring(mask, r=RING_PX):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k)
    ring = np.clip(dil.astype(np.int16) - mask.astype(np.int16), 0, 1).astype(np.uint8)
    if ring.sum() == 0:
        ring = (1 - mask.astype(np.uint8))
    return ring

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

# -------- TN3K train pool and THE SAME support episode as Phases 3/3c/3e ----------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files_generic(image_dir)}
    msks = {p.stem: p for p in image_files_generic(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"id": k, "image": str(imgs[k]), "mask": str(msks[k]),
                          "dataset": dataset, "split": split} for k in common])

def image_files_generic(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

tn3k = pair_images_masks(ART["tn3k_root"]/"trainval-image", ART["tn3k_root"]/"trainval-mask",
                         "TN3K", "trainval")
# canonical Phase-3c chain: sha256-dedup, then seed-42 15% val carve-out BEFORE support pick
assert len(tn3k) > 2700, f"TN3K trainval raw pool drift: {len(tn3k)} (canonical 2879)"
tn3k["sha256"] = tn3k["image"].apply(sha256_file)
tn3k = tn3k.sort_values("id", kind="stable").drop_duplicates("sha256", keep="first")
_rs = np.random.RandomState(SEED)
_perm = _rs.permutation(len(tn3k))
_val_ids = set(tn3k.iloc[_perm[: int(round(0.15 * len(tn3k)))]].id)
tn3k["split"] = np.where(tn3k.id.isin(_val_ids), "val", "train")
tn3k_train = tn3k[tn3k.split == "train"].reset_index(drop=True)
assert len(tn3k_train) > 2000, f"TN3K support pool drift after dedup/split: {len(tn3k_train)}"
print(f"TN3K pool (canonical 3c chain): {len(tn3k)} dedup pairs -> train {len(tn3k_train)}")

def tn3k_support_rows(kmax=SEG_K, seed=SEED + 10):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(tn3k_train))[:kmax]
    return tn3k_train.iloc[idx].reset_index(drop=True)

SUP_ROWS = tn3k_support_rows()
_arch = json.loads(Path(ART["p3_summary"]).read_text())["config"]["support_episode_ids"]
assert list(SUP_ROWS.id) == _arch, "support episode mismatch vs Phase 3 archive - seed drift!"
print(f"support episode reproduced exactly ({SEG_K} ids match Phase 3 archive); pool n={len(tn3k_train)}")

def load_tn3k_item(row):
    img = letterbox_image(read_gray(row["image"]).astype(np.float32))
    msk = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    return img, msk

## 2. TN5000-test manifest with expert boxes — canonical Phase-3 chain
Pascal-VOC XML parsing with the documented union rule (union of deciding boxes;
a **malignant** object decides the label; else all objects), official ImageSets
split files, within-dataset dedup (train>val>test), cross-dataset overlap removal
vs the TN3K train pool. Hard legacy gates: **test n = 950, malignant = 693**;
canonical id order taken from the archived Phase-3 feature bank when ids are
present there.

In [ ]:
import xml.etree.ElementTree as ET
def parse_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, names = [], []
    for ob in root.findall("object"):
        nm = (ob.findtext("name") or "").strip().lower()
        bb = ob.find("bndbox")
        if bb is None:
            continue
        try:
            box = [float(bb.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")]
        except (TypeError, ValueError):
            continue
        boxes.append(box); names.append(nm)
    return boxes, names

BENIGN, MALIGNANT = {"0", "benign"}, {"1", "malignant"}
def xml_to_row(xml_path):
    boxes, names = parse_xml(xml_path)
    if not boxes:
        return None
    mal = [i for i, n in enumerate(names) if n in MALIGNANT and n not in BENIGN]
    label = 1 if mal else 0
    use = [boxes[i] for i in (mal if mal else range(len(boxes)))]   # union of deciding boxes
    x0 = min(b[0] for b in use); y0 = min(b[1] for b in use)
    x1 = max(b[2] for b in use); y1 = max(b[3] for b in use)
    return {"label": label, "n_obj": len(boxes), "box": [x0, y0, x1, y1]}

t5_img = {p.stem: p for p in image_files_generic(ART["tn5000_root"] / "JPEGImages")}
xml_dir = ART["tn5000_root"] / "Annotations"
xmls = {p.stem: p for p in xml_dir.glob("*.xml")} if xml_dir.exists() else {}
common = sorted(set(t5_img) & set(xmls))
assert common, "TN5000: no JPEG/XML name overlap - check mount paths."
rows, skipped = [], 0
for k in common:
    r = xml_to_row(xmls[k])
    if r is None:
        skipped += 1; continue
    rows.append({"dataset": "TN5000", "id": k, "image": str(t5_img[k]), **r})
tn5000 = pd.DataFrame(rows)
print(f"TN5000: {len(tn5000)} usable (skipped {skipped} XML with no valid box) | "
      f"multi-object imgs {(tn5000.n_obj > 1).sum()} | prevalence malignant "
      f"{tn5000.label.mean():.3f} (n={int(tn5000.label.sum())})")

main_dir = ART["tn5000_root"] / "ImageSets" / "Main"
def read_split_file(stem):
    for cand in (main_dir / f"{stem}.txt", main_dir / f"{stem.upper()}.txt"):
        if cand.exists():
            ids = []
            for ln in cand.read_text().splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                ids.append(Path(ln.split()[0]).stem)
            return ids, str(cand)
    return None, None
splits, used_files = {}, {}
for stem in ["train", "val", "test"]:
    ids, src = read_split_file(stem)
    if ids is not None:
        splits[stem] = ids; used_files[stem] = src
tn5000["split"] = None
if len(splits) == 3:
    for stem, ids in splits.items():
        tn5000.loc[tn5000.id.isin(ids), "split"] = stem
    print("official split files:", used_files)
else:
    print("WARNING: official split files not found - deterministic fallback 3500/500/1000")
    perm = np.random.RandomState(SEED).permutation(len(tn5000))
    idx = tn5000.index.to_numpy()[perm]
    tn5000.loc[idx[:3500], "split"] = "train"
    tn5000.loc[idx[3500:4000], "split"] = "val"
    tn5000.loc[idx[4000:], "split"] = "test"
_uncov = int(tn5000.split.isna().sum())
if _uncov:
    print(f"WARNING: {_uncov} TN5000 rows not covered by the split files - dropping them "
          f"(unlabelled for our purposes); coverage issue to report in the audit log")
    tn5000 = tn5000[tn5000.split.notna()].copy()
tn5000 = tn5000.reset_index(drop=True)

tn5000["sha256"] = tn5000["image"].apply(sha256_file)
tn5000["_p"] = tn5000["split"].map({"train": 0, "val": 1, "test": 2})
before = len(tn5000)
tn5000 = tn5000.sort_values(["_p", "id"], kind="stable").drop_duplicates("sha256", keep="first")
tn5000 = tn5000.drop(columns="_p").reset_index(drop=True)
print(f"TN5000 dedup: {before} -> {len(tn5000)} (dropped {before - len(tn5000)} exact-duplicate images)")
ov = set(tn5000.sha256) & set(tn3k_train.sha256)
if ov:
    tn5000 = tn5000[~tn5000.sha256.isin(ov)].reset_index(drop=True)
print(f"cross-dataset overlap TN5000 vs TN3K-train: {len(ov)} (removed from TN5000)")

SPLITS = {s: tn5000[tn5000.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}
for s, d in SPLITS.items():
    print(f"  {s:5s} n={len(d):5d}  malignant={int(d.label.sum()):4d} ({d.label.mean():.3f})")
TN5K_TEST = SPLITS["test"]
assert len(TN5K_TEST) == 950, f"TN5000-test drift: {len(TN5K_TEST)} (locked 950)"
assert int(TN5K_TEST.label.sum()) == 693, f"TN5000-test positives drift (locked 693)"

# canonical ordering: archived feature-bank id order (falls back to manifest order with
# the ERM sanity AUROC gate as the alignment tripwire)
_zt = np.load(ART["tn5k_ftest"], allow_pickle=True)
print("feature bank keys:", _zt.files)
if "ids" in _zt.files:
    _zids = [Path(str(x)).stem for x in _zt["ids"].tolist()]
    _pos = {k: i for i, k in enumerate(_zids)}
    assert set(TN5K_TEST.id) == set(_zids), "TN5000-test ids != archived feature-bank ids"
    TN5K_TEST = TN5K_TEST.set_index("id").loc[_zids].reset_index()
y_t5 = _zt["labels"].astype(int).ravel()
assert len(y_t5) == len(TN5K_TEST) and (TN5K_TEST.label.to_numpy() == y_t5).all(), \
    "label order mismatch vs archived feature bank"
print(f"TN5K_TEST canonical: n={len(TN5K_TEST)}, malignant={int(y_t5.sum())}; "
      f"order {'archived-id' if 'ids' in _zt.files else 'manifest (gated by ERM sanity)'}")


## 3. Frozen machinery II — Phase-1 ResNet34-UNet trunk taps, support-conditioned
prototype segmentation (Phase-2 fusion decoder), fixed support prototypes.
Byte-identical to Phase 5.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class JointUNet(nn.Module):
    """Phase-1 ResNet34-UNet trunk (state-dict compatible), used FROZEN here."""
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e0": e0, "e1": e1, "e2": e2, "e3": e3, "e4": e4}
    def decode(self, f):
        d = self.up1(self.up2(self.up3(self.up4(f["e4"], f["e3"]), f["e2"]), f["e1"]), f["e0"])
        return self.head(d)
    def forward(self, x):
        f = self.features(x)
        return self.decode(f), f

def load_ckpt(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def build_frozen_trunk():
    trunk = JointUNet().to(DEVICE)
    _ck = load_ckpt(ART["ckpt_phase1"])
    trunk.load_state_dict(_ck["model"], strict=True)
    trunk.eval()
    for p in trunk.parameters():
        p.requires_grad_(False)
    print(f"  frozen Phase-1 trunk (which={_ck.get('which')}, val Dice={_ck.get('val_dice'):.4f})")
    return trunk

def encode(trunk, x):
    return trunk.features(x)

def masked_proto(feats, masks):
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    return -(feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)

class FusionDecoder(nn.Module):
    def __init__(self, cin=2 * len(PROTO_SCALES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits_384(que_feats, pf_pb_per_scale, decoder):
    return F.interpolate(decoder(score_maps(que_feats, pf_pb_per_scale)),
                         size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

trunk_frozen = build_frozen_trunk()
decoder = FusionDecoder().to(DEVICE)
_dk = load_ckpt(ART["ckpt_decoder"])
decoder.load_state_dict(_dk["decoder"], strict=True)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

_sup = [load_tn3k_item(r) for _, r in SUP_ROWS.iterrows()]
sup_x = torch.stack([to_tensor(i) for i, _ in _sup]).to(DEVICE)
sup_m = torch.stack([torch.from_numpy(m[None]) for _, m in _sup]).float().to(DEVICE)
with torch.no_grad():
    SF = encode(trunk_frozen, sup_x)
PF_PB = {name: masked_proto(SF[name], sup_m) for name in PROTO_SCALES}
print("decoder loaded | fixed support prototypes ready (3 scales)")

## 4. Locked heads, models, and frozen-row integrity gates
3c-retrained linear heads (`w, b, mu, sd` from `linear_heads_retrained.json`), ERM
checkpoint, saved Phase-4 JOINT arms. Gates recomputed from the **archived** probe
probabilities exactly as in Phase 5 (ERM sanity 0.9401; R1p integrity 0.8889 within the
0.002 tolerance of the locked 0.8902). Live-recompute gates happen in §5.

In [ ]:
def masked_gap(feat, msk01):
    """Hard per-image masked GAP (locked Phase-3 convention; nearest downsample)."""
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)            # (B,1) per-image masked area
    return (feat * m).sum(dim=(2, 3)) / den           # (B, C) per-image descriptor

def dice_np(a, b, eps=1e-7):
    a = a.astype(np.uint8); b = (np.asarray(b) > 0).astype(np.uint8)
    return float((2 * (a & b).sum() + eps) / (a.sum() + b.sum() + eps))

ERMJ = json.loads(Path(ART["erm_probs"]).read_text())
P3S  = json.loads(Path(ART["p3_summary"]).read_text())
LINR = json.loads(Path(ART["lin_retr"]).read_text())
JP   = json.loads(Path(ART["joint_probs"]).read_text())

def labels_np(split):
    z = np.load(ART["tn5k_ftest"], allow_pickle=True) if split == "test" else np.load(ART["tn5k_fval"], allow_pickle=True)
    return z["labels"].astype(int)

y_srct = labels_np("test").astype(int).ravel()
assert (y_srct == y_t5).all(), "archived label order != canonical TN5K_TEST order"
erm_ref = roc_auc_score(y_srct, np.array(ERMJ["probs"]["test"]).ravel())
assert abs(erm_ref - ERM_REF_AUROC) <= GATE_TOL_AUROC, f"ERM sanity failed: {erm_ref:.4f}"
print(f"ERM TN5000 test AUROC sanity: {erm_ref:.4f} (ref {ERM_REF_AUROC}) OK")

r1p_ref = roc_auc_score(y_srct, np.array(LINR["R1p_lesion"]["probs"]["test"]).ravel())
assert abs(r1p_ref - R1P_REF_AUROC) <= GATE_TOL_AUROC, f"R1p integrity failed: {r1p_ref:.4f}"
print(f"R1p TN5000 test AUROC integrity: {r1p_ref:.4f} (ref {R1P_REF_AUROC}) OK")

def lin_apply(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    z = ((X.astype(np.float64) - mu) / sd) @ w.T + b                       # locked 3c/4 convention
    return (1 / (1 + np.exp(-z.ravel()))).astype(float)

def lin_logits(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    return (((X.astype(np.float64) - mu) / sd) @ w.T + b).ravel()          # same formula, raw logit

import torchvision.models as tvm
erm = tvm.resnet34(weights=None); erm.fc = nn.Linear(erm.fc.in_features, 1)
erm.load_state_dict(load_ckpt(ART["erm_ckpt"])["model"]); erm = erm.to(DEVICE).eval()
for p in erm.parameters():
    p.requires_grad_(False)

def build_head_768():
    return nn.Linear(768, 1).to(DEVICE)

def load_joint_arm(tag):
    st = load_ckpt(ART[tag])
    trk = JointUNet().to(DEVICE); hd = build_head_768()
    trk.load_state_dict(st["trunk"], strict=True); hd.load_state_dict(st["head"], strict=True)
    trk.eval(); hd.eval()
    for p in list(trk.parameters()) + list(hd.parameters()):
        p.requires_grad_(False)
    return trk, hd, st

ARMG_TRK, ARMG_HEAD, ARMG_STATE = load_joint_arm("joint_g")
ARMI_TRK, ARMI_HEAD, ARMI_STATE = load_joint_arm("joint_img")
assert ARMG_STATE["pool_mode"] == "lesion" and ARMI_STATE["pool_mode"] == "image"
for nm, st in [("JOINT-g", ARMG_STATE), ("JOINT-img", ARMI_STATE)]:
    print(f"  {nm}: saved epoch {st['epoch']} | val AUROC {st['val_auroc']:.4f} | protocol v2")

assert not any(p.requires_grad for m in [trunk_frozen, decoder, erm, ARMG_TRK, ARMI_TRK,
               ARMG_HEAD, ARMI_HEAD] for p in m.parameters()), "a frozen model is trainable"
print("all heads/models loaded; frozen-row gates OK (archive-recomputed)")


## 5. XAI core — pure-numpy (sandbox-audited) engines
ELA is **exact** for masked/plain-GAP linear heads: descriptor
$d_c = \sum_x M(x)F_c(x)/\max(|M|,1)$, logit
$z = \sum_j (w_j/s_j)d_j + b^{\prime}$, so the per-pixel attribution
$A_s(x) = M_s(x)\sum_c \tilde w_c F_{s,c}(x)/\max(|M_s|,1)$ sums to $z$ exactly.
Regions: `les` (post-fallback predicted mask), `ring` (15 px), `img` (whole image).
Native masks are produced with the same nearest downsample the descriptor pipeline uses
(torch `F.interpolate(mode="nearest")`); every downstream CAM metric operates on the
ReLU→ℓ1-normalised 384² map.

In [ ]:
# ---------- Phase-3 box geometry (verbatim conventions) ----------
def letterbox_geom(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    t, l = (size - nh) // 2, (size - nw) // 2
    return {"s": s, "t": t, "l": l, "nh": nh, "nw": nw}

def box_to_384(box, geom):
    s, t, l = geom["s"], geom["t"], geom["l"]
    x0, y0, x1, y1 = box
    xa, ya = x0 * s + l, y0 * s + t
    xb, yb = x1 * s + l, y1 * s + t
    return [xa, ya, xb, yb]

def box_mask(box384, size=IMG_SIZE):
    m = np.zeros((size, size), np.uint8)
    x0, y0 = max(0, int(np.floor(box384[0]))), max(0, int(np.floor(box384[1])))
    x1, y1 = min(size, int(np.ceil(box384[2]))),  min(size, int(np.ceil(box384[3])))
    if x1 > x0 and y1 > y0:
        m[y0:y1, x0:x1] = 1
    return m

# ---------- CAM post-processing convention (pre-registered: ReLU -> l1) ----------
def relu_l1(cam):
    a = np.maximum(np.asarray(cam, np.float64), 0.0)
    s = a.sum()
    return (a / s if s > 0 else a), bool(s > 0)

def energy_in(cam01, m01):
    return float(cam01[np.asarray(m01) > 0].sum())

def cam_entropy01(cam01):
    p = cam01[cam01 > 0]
    return float(-(p * np.log(p)).sum() / np.log(cam01.size)) if p.size else float("nan")

# ---------- exact linear attribution (ELA) ----------
def LAYOUT768(region):
    return [(region, "e3", 256), (region, "e4", 512)]
ELA_LAYOUT = {"R0_image": LAYOUT768("img"),
              "R1p_lesion": LAYOUT768("les"),
              "R2p_lesion_ring": [("les", "e3", 256), ("les", "e4", 512),
                                  ("ring", "e3", 256), ("ring", "e4", 512)]}

def head_blocks_linr(rep):
    """3c head -> [(region, scale, w~/per-channel)], offset b'; locked standardisation."""
    h = LINR[rep]["head"]
    w = np.asarray(h["w"], np.float64).ravel(); b = float(np.asarray(h["b"], np.float64).ravel()[0])
    mu = np.asarray(h["mu"], np.float64).ravel(); sd = np.asarray(h["sd"], np.float64).ravel()
    assert np.all(sd > 0), f"{rep}: zero sd in locked head"
    out, off = [], 0
    for (region, scale, c) in ELA_LAYOUT[rep]:
        out.append((region, scale, w[off:off + c] / sd[off:off + c])); off += c
    assert off == w.size, f"{rep}: head length {w.size} != descriptor layout {off}"
    return out, b - float((w * mu / sd).sum())

def head_blocks_raw(w, b, layout):
    """JOINT heads: no standardisation (raw descriptors; locked Phase-4 convention)."""
    w = np.asarray(w, np.float64).ravel()
    out, off = [], 0
    for (region, scale, c) in layout:
        out.append((region, scale, w[off:off + c].copy())); off += c
    assert off == w.size, f"raw head length {w.size} != layout {off}"
    return out, float(b)

def ela_native(feats64, masks_native, blocks):
    """feats64: {"e3": (256,24,24), "e4": (512,12,12)} float64;
       masks_native: {(region, scale): (m (Hs,Ws) float64, den)} prepared by the caller;
       returns {scale: (Hs,Ws)} attribution whose total sums to z - offset (exact)."""
    out = {}
    for (region, scale, wv) in blocks:
        m, den = masks_native[(region, scale)]
        amap = m * np.tensordot(wv, feats64[scale], axes=([0], [0])) / den
        out[scale] = amap if scale not in out else out[scale] + amap
    return out

def ela_sum(native):
    return float(sum(float(m.sum()) for m in native.values()))

def cam384(native, size=IMG_SIZE):
    return sum(cv2.resize(m, (size, size), interpolation=cv2.INTER_LINEAR)
               for m in native.values())

# ---------- faithfulness helpers ----------
def order_pixels(cam01):
    """Descending energy; ties resolved by lowest flat index (pre-declared, stable)."""
    return np.argsort(-cam01.ravel(), kind="stable")

def apply_step(raw01, base01, order, k, steps, direction):
    n = int(round(raw01.size * k / steps))
    idx = order[:n]
    if direction == "delete":
        out = raw01.copy(); out.ravel()[idx] = base01.ravel()[idx]
    elif direction == "insert":
        out = base01.copy(); out.ravel()[idx] = raw01.ravel()[idx]
    else:
        raise ValueError(direction)
    return out

def trapz_area(y, dx):
    fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return float(fn(np.asarray(y, np.float64), dx=dx))

# ---------- paired bootstrap (same seeded discipline as families G/H/J) ----------
def paired_mean_boot(d, n=N_BOOT_K, seed=SEED):
    d = np.asarray(d, np.float64)
    rng = np.random.RandomState(seed); N = d.size; bs = []
    for _ in range(n):
        bs.append(d[rng.randint(0, N, N)].mean())
    bs = np.array(bs)
    return bs.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), \
        min(2 * min((bs <= 0).mean(), (bs >= 0).mean()), 1.0), len(bs)

def spearman_safe(a, b):
    a = np.asarray(a, np.float64).ravel(); b = np.asarray(b, np.float64).ravel()
    if a.size < 3 or np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    return float(spearmanr(a, b).statistic)


## 6. Torch-side engines (thin wrappers over the pure core)
Grad-CAM at `layer4` (hook-captured activations, input-grad enabled — frozen weights),
Eigen-CAM (first principal component of layer4 activations, sign fixed to positive mass),
Score-CAM (top-64 channels by |Grad-CAM α|, raw-prob channel scores), native-region-mask
preparation with the pipeline's own nearest downsample, and per-model canvas scorers used
by the faithfulness battery (masks held FIXED at the original prediction).

In [ ]:
ACTS = {}
def _hook(name):
    return lambda m, i, o: ACTS.__setitem__(name, o)

def erm_forward_cam(x1):
    """x1 (1,3,384,384) DEVICE fp32. Returns (prob, gradcam (12,12) float64, acts (512,12,12))."""
    h = erm.layer4.register_forward_hook(_hook("erm_e4"))
    try:
        with torch.no_grad():
            prob = float(torch.sigmoid(erm(x1)).item())
        xg = x1.clone().requires_grad_(True)
        erm.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logit = erm(xg)
            g = torch.autograd.grad(logit.sum(), ACTS["erm_e4"])[0]
    finally:
        h.remove()
    a = ACTS["erm_e4"][0].detach().cpu().numpy().astype(np.float64)
    gg = g[0].cpu().numpy().astype(np.float64)
    alpha = gg.mean(axis=(1, 2), keepdims=True)
    return prob, np.maximum((alpha * a).sum(0), 0.0), a

def cam_eigencam(a):
    """a (512,h,w) float64 -> (h,w) Eigen-CAM (no centering; positive-mass sign)."""
    A = a.reshape(a.shape[0], -1)
    u, s, vt = np.linalg.svd(A, full_matrices=False)
    proj = (u[:, 0][:, None] * A).sum(0).reshape(a.shape[1:])
    if proj.sum() < 0:
        proj = -proj
    return np.maximum(proj, 0.0)

def cam_scorecam(x1, a, top_idx):
    """Raw-prob channel scores on the normalised input (pre-declared convention)."""
    H, W = x1.shape[-2:]
    ups = np.stack([cv2.resize(a[c], (W, H), interpolation=cv2.INTER_LINEAR) for c in top_idx])
    ups = ups / (ups.max(axis=(1, 2), keepdims=True) + 1e-8)
    xb = x1[0].detach().cpu().numpy().astype(np.float32)
    xs = torch.from_numpy((xb[None] * ups[:, None]).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        sc = torch.sigmoid(erm(xs)).squeeze(1).cpu().numpy().astype(np.float64)
    return np.maximum((sc[:, None, None] * a[top_idx]).sum(0), 0.0)

FEAT_HW = {"e3": (IMG_SIZE // 16, IMG_SIZE // 16), "e4": (IMG_SIZE // 32, IMG_SIZE // 32)}
def masks_native_np(regions384, sizes=FEAT_HW):
    """Nearest downsample identical to masked_gap; per-(region, scale) (mask, den)."""
    out = {}
    for rgn, m384 in regions384.items():
        t = torch.from_numpy(np.asarray(m384, np.float32)[None, None])
        for scale, hw in sizes.items():
            m = F.interpolate(t, size=hw, mode="nearest").numpy()[0, 0].astype(np.float64)
            out[(rgn, scale)] = (m, max(float(m.sum()), 1.0))
    return out

def stack_raw(canvases):
    return torch.stack([to_tensor(c.astype(np.float32)) for c in canvases]).to(DEVICE)

def probs_lin_rep(rep, x, fix_mask01):
    """Frozen trunk + FIXED mask -> descriptor -> locked 3c head probs (B,)."""
    with torch.no_grad():
        f = encode(trunk_frozen, x)
        m = torch.from_numpy(fix_mask01[None, None].astype(np.float32)).to(DEVICE).expand(x.shape[0], 1, -1, -1)
        if rep == "R0_image":
            d = torch.cat([f[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        elif rep == "R1p_lesion":
            d = torch.cat([masked_gap(f[nm], m) for nm in FEAT_SCALES], 1)
        elif rep == "R2p_lesion_ring":
            ring = torch.from_numpy(make_ring(fix_mask01)[None, None].astype(np.float32)).to(DEVICE) \
                       .expand(x.shape[0], 1, -1, -1)
            a = torch.cat([masked_gap(f[nm], m) for nm in FEAT_SCALES], 1)
            b = torch.cat([masked_gap(f[nm], ring) for nm in FEAT_SCALES], 1)
            d = torch.cat([a, b], 1)
        else:
            raise ValueError(rep)
    return lin_apply(rep, d.cpu().numpy())

def probs_joint(arm, x, fix_mask01=None):
    trk, hd = (ARMG_TRK, ARMG_HEAD) if arm == "g" else (ARMI_TRK, ARMI_HEAD)
    with torch.no_grad():
        f = trk.features(x)
        if arm == "g":
            m = torch.from_numpy(fix_mask01[None, None].astype(np.float32)).to(DEVICE) \
                      .expand(x.shape[0], 1, -1, -1)
            d = torch.cat([masked_gap(f[nm], m) for nm in FEAT_SCALES], 1)
        else:
            d = torch.cat([f[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        p = torch.sigmoid(hd(d.float())).squeeze(1)
    return p.cpu().numpy().astype(float)

def probs_erm(x):
    with torch.no_grad():
        return torch.sigmoid(erm(x)).squeeze(1).cpu().numpy().astype(float)


## 7. TN5000-test panel — forwards, ELA, Grad-CAM family, GT-box plausibility
One pass over the canonical 950 test images: frozen masks (post-fallback), descriptors
and locked-head probabilities for R0/R1p/R2p; JOINT-g/JOINT-img probabilities and ELA;
ERM probabilities + Grad-CAM (+ Eigen-CAM; Score-CAM on a seeded 150-image subsample).
Live-recompute gates on this panel: ERM AUROC vs 0.9401; recomputed R1p AUROC vs the
archived 0.8889; JOINT lineage vs the archived Phase-4 probabilities
(archive-matched autocast policy; the fp32-panel deviation is recorded as a documented
numeric-policy note, not gated); ELA exactness on a
seeded 32-image probe. Plausibility columns per image: in-box share, localisation ratio
(share ÷ box-area share), pointing-game hit — plus descriptive mask/ring energy and CAM
entropy.

In [ ]:
t_panel = time.time()
ZL = np.load(ART["tn5k_ftest"], allow_pickle=True)
rng_probe = np.random.RandomState(SEED + 3)
PROBE_IDX = set(rng_probe.choice(len(TN5K_TEST), size=ELA_PROBE_N, replace=False).tolist())
rng_sc = np.random.RandomState(SEED + 4)
SC_IDX = set(rng_sc.choice(len(TN5K_TEST), size=SCORECAM_N, replace=False).tolist())
rng_mont = np.random.RandomState(SEED + 5)
MONT_IDX = set(rng_mont.choice(len(TN5K_TEST), size=MONTAGE_N, replace=False).tolist())

BLOCKS_LIN = {rep: head_blocks_linr(rep) for rep in ["R0_image", "R1p_lesion", "R2p_lesion_ring"]}
_wg = ARMG_HEAD.weight.detach().cpu().numpy().astype(np.float64).ravel()
_bg = float(ARMG_HEAD.bias.detach().cpu().numpy().astype(np.float64)[0])
BLOCKS_JG = head_blocks_raw(_wg, _bg, LAYOUT768("les"))
_wi = ARMI_HEAD.weight.detach().cpu().numpy().astype(np.float64).ravel()
_bi = float(ARMI_HEAD.bias.detach().cpu().numpy().astype(np.float64)[0])
BLOCKS_JI = head_blocks_raw(_wi, _bi, LAYOUT768("img"))

N5 = len(TN5K_TEST)
IDS, Y = TN5K_TEST.id.tolist(), TN5K_TEST.label.to_numpy().astype(int)
DESC = {rep: np.zeros((N5, 768 if rep != "R2p_lesion_ring" else 1536), np.float32)
        for rep in ["R0_image", "R1p_lesion", "R2p_lesion_ring"]}
P_JG, P_JI, P_ERM = np.zeros(N5), np.zeros(N5), np.zeros(N5)
EMPTY_FLAG, MASK_AREA = np.zeros(N5, np.int64), np.zeros(N5)
BOXM = np.zeros((N5, IMG_SIZE, IMG_SIZE), np.uint8)
MET = {k: np.full(N5, np.nan) for k in
       ["boxfrac", "r1p_inbox", "r1p_ratio", "r1p_hit", "gc_inbox", "gc_ratio", "gc_hit",
        "eig_inbox", "eig_ratio", "eig_hit", "sc_inbox", "sc_ratio", "sc_hit",
        "r0_inbox", "r2p_inbox", "jg_inbox", "ji_inbox", "gc_inmask", "gc_inring",
        "r1p_ent", "gc_ent"]}
ELA_DEV = {rep: 0.0 for rep in ["R0_image", "R1p_lesion", "R2p_lesion_ring", "JG", "JI"]}
MONT = {}
for i0 in range(0, N5, BATCH):
    chunk = TN5K_TEST.iloc[i0:i0 + BATCH]
    raws, bms = [], []
    for _, r in chunk.iterrows():
        img_raw = read_gray(r["image"]).astype(np.float32)
        raws.append(letterbox_image(img_raw))
        bms.append(box_mask(box_to_384(r["box"], letterbox_geom(img_raw))))
    x = torch.stack([to_tensor(r_) for r_ in raws]).to(DEVICE)
    with torch.no_grad():
        feats = encode(trunk_frozen, x)
        prob_seg = torch.sigmoid(predict_logits_384(feats, PF_PB, decoder))
        lg, fg = ARMG_TRK(x)
        pmg = (torch.sigmoid(lg.float()) >= PRED_THRESH).float()
        fi = ARMI_TRK.features(x)
        P_ERM[i0:i0 + len(chunk)] = torch.sigmoid(erm(x)).squeeze(1).float().cpu().numpy()
    pm = (prob_seg >= PRED_THRESH).float()
    for bi in range(len(chunk)):
        gi = i0 + bi
        if pm[bi].sum() < 4:
            pm[bi] = 1.0; EMPTY_FLAG[gi] = 1
        if pmg[bi].sum() < 4:
            pmg[bi] = 1.0
    ring = torch.stack([torch.from_numpy(make_ring(pm[bi, 0].cpu().numpy().astype(np.uint8))[None])
                        for bi in range(len(chunk))]).float().to(DEVICE)
    with torch.no_grad():
        d0 = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        d1 = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
        d2a = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
        d2b = torch.cat([masked_gap(feats[nm], ring) for nm in FEAT_SCALES], 1)
        dg = torch.cat([masked_gap(fg[nm], pmg) for nm in FEAT_SCALES], 1)
        di = torch.cat([fi[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        P_JG[i0:i0 + len(chunk)] = torch.sigmoid(ARMG_HEAD(dg.float())).squeeze(1).float().cpu().numpy()
        P_JI[i0:i0 + len(chunk)] = torch.sigmoid(ARMI_HEAD(di.float())).squeeze(1).float().cpu().numpy()
    DESC["R0_image"][i0:i0 + len(chunk)] = d0.float().cpu().numpy()
    DESC["R1p_lesion"][i0:i0 + len(chunk)] = d1.float().cpu().numpy()
    DESC["R2p_lesion_ring"][i0:i0 + len(chunk)] = torch.cat([d2a, d2b], 1).float().cpu().numpy()
    f64 = {nm: feats[nm].detach().cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    fg64 = {nm: fg[nm].detach().cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    fi64 = {nm: fi[nm].detach().cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    for nm in FEAT_SCALES:
        assert feats[nm].shape[-2:] == FEAT_HW[nm], f"feature-map size drift {nm}: {feats[nm].shape}"
    pm_np = pm.cpu().numpy(); pmg_np = pmg.cpu().numpy(); ring_np = ring.cpu().numpy()
    for bi in range(len(chunk)):
        gi = i0 + bi
        bm = bms[bi]; BOXM[gi] = bm
        MASK_AREA[gi] = float(pm_np[bi, 0].mean())
        boxfrac = max(bm.mean(), 1e-9); MET["boxfrac"][gi] = boxfrac
        regions = {"les": pm_np[bi, 0], "ring": ring_np[bi, 0],
                   "img": np.ones((IMG_SIZE, IMG_SIZE), np.uint8)}
        mn = masks_native_np(regions)
        f1 = {nm: f64[nm][bi] for nm in FEAT_SCALES}
        nat = {}
        for rep, (blocks, off) in BLOCKS_LIN.items():
            nat[rep] = ela_native(f1, mn, blocks)
        regions_g = {"les": pmg_np[bi, 0]}
        mn_g = masks_native_np({**regions_g, "img": regions["img"]})
        fg1 = {nm: fg64[nm][bi] for nm in FEAT_SCALES}
        fi1 = {nm: fi64[nm][bi] for nm in FEAT_SCALES}
        nat["JG"] = ela_native(fg1, mn_g, BLOCKS_JG[0])
        nat["JI"] = ela_native(fi1, mn_g, BLOCKS_JI[0])
        if gi in PROBE_IDX:
            dev = {}
            for rep in ["R0_image", "R1p_lesion", "R2p_lesion_ring"]:
                z = lin_logits(rep, DESC[rep][gi:gi + 1])[0]
                dev[rep] = abs(ela_sum(nat[rep]) + BLOCKS_LIN[rep][1] - z)
            dg64 = dg[bi].detach().cpu().numpy().astype(np.float64).ravel()
            di64 = di[bi].detach().cpu().numpy().astype(np.float64).ravel()
            zg = float(dg64 @ np.concatenate([b[2] for b in BLOCKS_JG[0]]) + BLOCKS_JG[1])
            zi = float(di64 @ np.concatenate([b[2] for b in BLOCKS_JI[0]]) + BLOCKS_JI[1])
            dev["JG"] = abs(ela_sum(nat["JG"]) + BLOCKS_JG[1] - zg)
            dev["JI"] = abs(ela_sum(nat["JI"]) + BLOCKS_JI[1] - zi)
            for k, v in dev.items():
                ELA_DEV[k] = max(ELA_DEV[k], float(v))
        p_erm, gc_native, acts = erm_forward_cam(x[bi:bi + 1])
        cams = {"R1p": cam384(nat["R1p_lesion"]), "R0": cam384(nat["R0_image"]),
                "R2p": cam384(nat["R2p_lesion_ring"]),
                "JG": cam384(nat["JG"]), "JI": cam384(nat["JI"])}
        gc384 = cv2.resize(gc_native, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        gc01, _ = relu_l1(gc384)
        r01, _ = relu_l1(cams["R1p"])
        eig384 = cv2.resize(cam_eigencam(acts), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        e01, _ = relu_l1(eig384)
        MET["r1p_inbox"][gi] = float(bm[regions["les"] > 0].mean()) if regions["les"].sum() else 0.0
        MET["r1p_ratio"][gi] = MET["r1p_inbox"][gi] / boxfrac
        amr = np.unravel_index(int(np.argmax(r01)), r01.shape)
        MET["r1p_hit"][gi] = float(bm[amr] > 0)
        MET["gc_inbox"][gi] = energy_in(gc01, bm); MET["gc_ratio"][gi] = MET["gc_inbox"][gi] / boxfrac
        MET["eig_inbox"][gi] = energy_in(e01, bm); MET["eig_ratio"][gi] = MET["eig_inbox"][gi] / boxfrac
        am = np.unravel_index(int(np.argmax(gc01)), gc01.shape)
        MET["gc_hit"][gi] = float(bm[am] > 0)
        ame = np.unravel_index(int(np.argmax(e01)), e01.shape)
        MET["eig_hit"][gi] = float(bm[ame] > 0)
        if gi in SC_IDX:
            top_idx = np.argsort(-np.abs(acts.mean(axis=(1, 2))))[:SCORECAM_TOP]
            sc384 = cv2.resize(cam_scorecam(x[bi:bi + 1], acts, top_idx),
                               (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            s01, _ = relu_l1(sc384)
            MET["sc_inbox"][gi] = energy_in(s01, bm); MET["sc_ratio"][gi] = MET["sc_inbox"][gi] / boxfrac
            ams = np.unravel_index(int(np.argmax(s01)), s01.shape)
            MET["sc_hit"][gi] = float(bm[ams] > 0)
        for tag, key in [("r0", "R0"), ("r2p", "R2p"), ("jg", "JG"), ("ji", "JI")]:
            c01, _ = relu_l1(cams[key])
            MET[f"{tag}_inbox"][gi] = energy_in(c01, bm)
        MET["gc_inmask"][gi] = energy_in(gc01, regions["les"])
        MET["gc_inring"][gi] = energy_in(gc01, regions["ring"])
        MET["r1p_ent"][gi] = cam_entropy01(r01)
        MET["gc_ent"][gi] = cam_entropy01(gc01)
        if gi in MONT_IDX:
            MONT[gi] = {"raw": raws[bi], "bm": bm, "pm": regions["les"].astype(np.uint8),
                        "r1p": r01, "gc": gc01, "label": int(Y[gi]), "p_erm": float(p_erm)}
    done = min(i0 + BATCH, N5)
    if done % (BATCH * 25) < BATCH or done == N5:
        print(f"  tn5k panel: {done}/{N5} ({(time.time() - t_panel) / 60:.1f} min)")

# ---- live-recompute gates ----
erm_live = roc_auc_score(Y, P_ERM)
assert abs(erm_live - ERM_REF_AUROC) <= GATE_TOL_AUROC, f"ERM live sanity failed: {erm_live:.4f}"
r1p_live = roc_auc_score(Y, lin_apply("R1p_lesion", DESC["R1p_lesion"]))
print(f"LIVE gates: ERM {erm_live:.4f} (ref 0.9401) | R1p recompute {r1p_live:.4f} "
      f"(archived {r1p_ref:.4f}) | R0 {roc_auc_score(Y, lin_apply('R0_image', DESC['R0_image'])):.4f} | "
      f"R2p {roc_auc_score(Y, lin_apply('R2p_lesion_ring', DESC['R2p_lesion_ring'])):.4f}")
assert abs(r1p_live - r1p_ref) <= GATE_TOL_AUROC, f"R1p live integrity failed: {r1p_live:.4f}"
# ---- live JOINT lineage: archive-matched numeric policy ---------------------------
# The archived JP probabilities were evaluated under Phase 4's autocast policy
# (autocast trunk on CUDA; mask from the autocast seg probabilities; head on
# desc.float()). The panel battery stays pure fp32 (declared; ELA/faithfulness are
# self-consistent against the fp32 panel via the step-0 integrity asserts). The
# provenance tripwire therefore replicates the archive's policy byte-faithfully and
# gates at 1e-4 (expectation ~1e-6; the fp32 deviation is recorded as a note, NOT
# gated). A wrong weights file / wrong state would produce O(0.05-0.5) deviations,
# hundreds of times above this gate.
JG_ARCH = np.array(JP["test"]["JOINTg"]).ravel(); JI_ARCH = np.array(JP["test"]["JOINTimg"]).ravel()

@torch.no_grad()
def live_joint_probs_archive_policy(trk, hd, pool_mode, df):
    """Byte-faithful replica of Phase-4 eval_cls_probs (autocast trunk on CUDA)."""
    ps = []
    for i0 in range(0, len(df), BATCH):
        chunk = df.iloc[i0:i0 + BATCH]
        x = torch.stack([to_tensor(letterbox_image(read_gray(r["image"]).astype(np.float32)))
                         for _, r in chunk.iterrows()]).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits_seg, feats = trk(x)
            prob_seg = torch.sigmoid(logits_seg.float())
        if pool_mode == "lesion":
            pm = (prob_seg >= PRED_THRESH).float()
            for bi in range(len(x)):
                if pm[bi].sum() < 4:
                    pm[bi] = 1.0
            desc = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
        else:
            desc = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        ps.append(torch.sigmoid(hd(desc.float())).squeeze(1).float().cpu())
    return torch.cat(ps).numpy().ravel()

P_JG_AMP = live_joint_probs_archive_policy(ARMG_TRK, ARMG_HEAD, "lesion", TN5K_TEST)
P_JI_AMP = live_joint_probs_archive_policy(ARMI_TRK, ARMI_HEAD, "image", TN5K_TEST)
lin_g, lin_i = float(np.abs(P_JG_AMP - JG_ARCH).max()), float(np.abs(P_JI_AMP - JI_ARCH).max())
lin_g_fp32, lin_i_fp32 = float(np.abs(P_JG - JG_ARCH).max()), float(np.abs(P_JI - JI_ARCH).max())
print(f"JOINT lineage (archive-policy): JOINT-g max|dp|={lin_g:.2e} | JOINT-img {lin_i:.2e} "
      f"[fp32 panel note: g {lin_g_fp32:.2e} / i {lin_i_fp32:.2e}; note only]")
assert lin_g <= 1e-4 and lin_i <= 1e-4, \
    "joint lineage drift beyond 1e-4 under the archive-matched autocast policy"
for k, v in ELA_DEV.items():
    print(f"ELA exactness probe max|dev| {k}: {v:.2e}")
    assert v < ELA_EXACT_TOL, f"ELA exactness failed for {k}: {v}"
print(f"empty-mask fallbacks on panel: {int(EMPTY_FLAG.sum())}/950 "
      f"({100 * EMPTY_FLAG.mean():.1f}%) | mean mask area {MASK_AREA.mean():.4f}")

np.savez_compressed(OUTPUT_ROOT / "phase6_panel_tn5000.npz",
                    ids=np.array(IDS), labels=Y, boxm=BOXM.astype(np.uint8),
                    empty=EMPTY_FLAG, mask_area=MASK_AREA,
                    p_erm=P_ERM, p_jg=P_JG, p_ji=P_JI,
                    p_r1p=lin_apply("R1p_lesion", DESC["R1p_lesion"]),
                    p_r0=lin_apply("R0_image", DESC["R0_image"]),
                    p_r2p=lin_apply("R2p_lesion_ring", DESC["R2p_lesion_ring"]),
                    **{f"met_{k}": v for k, v in MET.items()})
import pickle
pickle.dump(MONT, open(OUTPUT_ROOT / "phase6_montage_store.pkl", "wb"))
print(f"[saved] phase6_panel_tn5000.npz | panel time {(time.time() - t_panel) / 60:.1f} min")


## 8. Montage (qualitative figure candidate)
Seeded 8-image panel: greyscale + expert box, predicted mask contour, R1p-ELA heat,
ERM Grad-CAM heat (ReLU→ℓ1 maps; same colour scale per row pair).

In [ ]:
fig, axes = plt.subplots(4, MONTAGE_N, figsize=(2.1 * MONTAGE_N, 8.6))
for j, gi in enumerate(sorted(MONT)):
    rec = MONT[gi]
    g = rec["raw"]; img3 = np.stack([g, g, g], -1)
    axes[0, j].imshow(img3, cmap="gray")
    import matplotlib.patches as mpatches
    if rec["bm"].sum() > 0:
        x0, y0 = np.where(rec["bm"])[1].min(), np.where(rec["bm"])[0].min()
        x1, y1 = np.where(rec["bm"])[1].max(), np.where(rec["bm"])[0].max()
        axes[0, j].add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                                fill=False, edgecolor="lime", lw=1.2))
    axes[0, j].set_title(f'{IDS[gi][:12]}\ny={rec["label"]} p(ERM)={rec["p_erm"]:.2f}', fontsize=6)
    ov = img3.copy(); ov[..., 0] = np.maximum(ov[..., 0], rec["pm"] * 0.9)
    axes[1, j].imshow(ov, cmap="gray"); axes[1, j].set_title("pred mask", fontsize=6)
    axes[2, j].imshow(g, cmap="gray"); axes[2, j].imshow(rec["r1p"], cmap="jet", alpha=0.55, vmin=0)
    axes[2, j].set_title("R1p ELA", fontsize=6)
    axes[3, j].imshow(g, cmap="gray"); axes[3, j].imshow(rec["gc"], cmap="jet", alpha=0.55, vmin=0)
    axes[3, j].set_title("ERM Grad-CAM", fontsize=6)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Phase-6 XAI montage (seeded panel): box → mask → R1p-ELA → ERM-GradCAM")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "phase6_cam_montage.png", dpi=130)
plt.show()
print("[saved] phase6_cam_montage.png")


## 9. Faithfulness — deletion/insertion under blur (TN5000-test)
Per image: baseline $\\tilde{x}$ = Gaussian blur ($\\sigma{=}8$). **Deletion** (primary):
replace the top-$k/8$ pixels by CAM order with $\\tilde{x}$, track the raw probability;
**insertion**: reverse. Controls: one seeded random ordering per image
(`RandomState(43)` stream). Masks FIXED at the original prediction (integrity: recomputed
step-0 probability must match the panel probability to 5e-6). Faithfulness gap:
deletion = AUC(random) − AUC(guided); insertion = AUC(guided) − AUC(random) — positive =
attribution-guided is better. Insertion on a seeded 300-image subset (descriptive).

In [ ]:
t_f = time.time()
rng_stream = np.random.RandomState(SEED + 1)          # pre-declared random-ordering stream
rng_ins = np.random.RandomState(SEED + 2)
INS_IDX = set(rng_ins.choice(len(TN5K_TEST), size=300, replace=False).tolist())
MODELS_K = ["R1p", "ERM", "JOINT-g", "JOINT-img"]
CUR = {m: {"del_guided": np.zeros(N5), "del_random": np.zeros(N5),
           "ins_guided": np.full(N5, np.nan), "ins_random": np.full(N5, np.nan),
           "gap_del": np.zeros(N5), "gap_ins": np.full(N5, np.nan)} for m in MODELS_K}
for gi in range(N5):
    r = TN5K_TEST.iloc[gi]
    raw01 = letterbox_image(read_gray(r["image"]).astype(np.float32))
    base01 = cv2.GaussianBlur(raw01, (0, 0), BLUR_SIGMA)
    with torch.no_grad():
        f1 = encode(trunk_frozen, stack_raw([raw01]))
        pm1 = (torch.sigmoid(predict_logits_384(f1, PF_PB, decoder)) >= PRED_THRESH).float()
        lg1, fg1 = ARMG_TRK(stack_raw([raw01]))
        pmg1 = (torch.sigmoid(lg1.float()) >= PRED_THRESH).float()
    fix = pm1[0, 0].cpu().numpy().astype(np.uint8)
    fixg = pmg1[0, 0].cpu().numpy().astype(np.uint8)
    if fix.sum() < 4:
        fix = np.ones_like(fix)
    if fixg.sum() < 4:
        fixg = np.ones_like(fixg)
    with torch.no_grad():
        fx = encode(trunk_frozen, stack_raw([raw01]))
        d1 = torch.cat([masked_gap(fx[nm], pm1) for nm in FEAT_SCALES], 1).cpu().numpy()
    p0 = {}
    p0["R1p"] = float(lin_apply("R1p_lesion", d1)[0])
    p0["JOINT-g"] = float(probs_joint("g", stack_raw([raw01]), fixg)[0])
    p0["JOINT-img"] = float(probs_joint("i", stack_raw([raw01]))[0])
    p_erm0, gc_native, _acts = erm_forward_cam(stack_raw([raw01]))
    p0["ERM"] = p_erm0
    assert abs(p0["R1p"] - float(lin_apply("R1p_lesion", DESC["R1p_lesion"][gi:gi + 1])[0])) < INTEGRITY_TOL, \
        f"R1p fixed-mask integrity drift at {gi}"
    assert abs(p0["ERM"] - float(P_ERM[gi])) < INTEGRITY_TOL, f"ERM integrity drift at {gi}"
    assert abs(p0["JOINT-g"] - float(P_JG[gi])) < INTEGRITY_TOL, f"JOINT-g integrity drift at {gi}"
    assert abs(p0["JOINT-img"] - float(P_JI[gi])) < INTEGRITY_TOL, f"JOINT-img integrity drift at {gi}"
    regions = {"les": fix, "img": np.ones_like(fix)}
    cam_r1p, _ = relu_l1(cam384(ela_native({nm: fx[nm][0].cpu().numpy().astype(np.float64)
                                            for nm in FEAT_SCALES},
                                           masks_native_np(regions), BLOCKS_LIN["R1p_lesion"][0])))
    cam_erm, _ = relu_l1(cv2.resize(gc_native, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR))
    with torch.no_grad():
        fg64_0 = {nm: fg1[nm][0].cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
        fi_0 = ARMI_TRK.features(stack_raw([raw01]))
    fi64_0 = {nm: fi_0[nm][0].detach().cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    cam_ji, _ = relu_l1(cam384(ela_native(fi64_0, masks_native_np({"img": regions["img"]}),
                                          BLOCKS_JI[0])))
    cams = {"R1p": cam_r1p, "ERM": cam_erm, "JOINT-g": cam_jg, "JOINT-img": cam_ji}
    order_r = rng_stream.permutation(raw01.size)
    for m in MODELS_K:
        og, orr = order_pixels(cams[m]), order_r
        del_g = [apply_step(raw01, base01, og, k, DEL_STEPS, "delete") for k in range(1, DEL_STEPS + 1)]
        del_r = [apply_step(raw01, base01, orr, k, DEL_STEPS, "delete") for k in range(1, DEL_STEPS + 1)]
        cg = stack_raw(del_g); cr = stack_raw(del_r)
        if m == "R1p":
            sg, sr = probs_lin_rep("R1p_lesion", cg, fix), probs_lin_rep("R1p_lesion", cr, fix)
        elif m == "ERM":
            sg, sr = probs_erm(cg), probs_erm(cr)
        elif m == "JOINT-g":
            sg, sr = probs_joint("g", cg, fixg), probs_joint("g", cr, fixg)
        else:
            sg, sr = probs_joint("i", cg), probs_joint("i", cr)
        a_g = trapz_area([p0[m]] + list(sg), 1 / DEL_STEPS)
        a_r = trapz_area([p0[m]] + list(sr), 1 / DEL_STEPS)
        CUR[m]["del_guided"][gi], CUR[m]["del_random"][gi] = a_g, a_r
        CUR[m]["gap_del"][gi] = a_r - a_g
        if gi in INS_IDX:
            ins_g = [apply_step(raw01, base01, og, k, DEL_STEPS, "insert") for k in range(1, DEL_STEPS + 1)]
            ins_r = [apply_step(raw01, base01, orr, k, DEL_STEPS, "insert") for k in range(1, DEL_STEPS + 1)]
            cg2, cr2 = stack_raw(ins_g), stack_raw(ins_r)
            if m == "R1p":
                tg, tr = probs_lin_rep("R1p_lesion", cg2, fix), probs_lin_rep("R1p_lesion", cr2, fix)
            elif m == "ERM":
                tg, tr = probs_erm(cg2), probs_erm(cr2)
            elif m == "JOINT-g":
                tg, tr = probs_joint("g", cg2, fixg), probs_joint("g", cr2, fixg)
            else:
                tg, tr = probs_joint("i", cg2), probs_joint("i", cr2)
            b0 = (probs_lin_rep("R1p_lesion", stack_raw([base01]), fix)[0] if m == "R1p"
                  else probs_erm(stack_raw([base01]))[0] if m == "ERM"
                  else probs_joint("g", stack_raw([base01]), fixg)[0] if m == "JOINT-g"
                  else probs_joint("i", stack_raw([base01]))[0])
            CUR[m]["ins_guided"][gi] = trapz_area([b0] + list(tg), 1 / DEL_STEPS)
            CUR[m]["ins_random"][gi] = trapz_area([b0] + list(tr), 1 / DEL_STEPS)
            CUR[m]["gap_ins"][gi] = CUR[m]["ins_guided"][gi] - CUR[m]["ins_random"][gi]
    if (gi + 1) % 100 == 0 or gi == N5 - 1:
        print(f"  faithfulness tn5k: {gi + 1}/{N5} ({(time.time() - t_f) / 60:.1f} min)")
SUM = []
for m in MODELS_K:
    ins = ~np.isnan(CUR[m]["gap_ins"])
    SUM.append({"model": m,
                "del_AUC guided": round(float(CUR[m]["del_guided"].mean()), 4),
                "del_AUC random": round(float(CUR[m]["del_random"].mean()), 4),
                "del gap": round(float(CUR[m]["gap_del"].mean()), 4),
                "ins gap (n=300)": round(float(CUR[m]["gap_ins"][ins].mean()), 4)})
FAITH_T5 = pd.DataFrame(SUM)
display(FAITH_T5)
np.savez_compressed(OUTPUT_ROOT / "phase6_curves_tn5000.npz",
                    **{f"{m}_{k}": v for m, d in CUR.items() for k, v in d.items()})
FAITH_T5.to_csv(OUTPUT_ROOT / "phase6_faithfulness_tn5000.csv", index=False)
print(f"[saved] curves + summary | faithfulness time {(time.time() - t_f) / 60:.1f} min")


## 10. ThyUS2Path external panel (label-free XAI metrics)
Seeded 600-case first-image panel from the **licensed** Phase-5 manifest (hard gates:
manifest n = 839, prevalence 0.6222 within 1e-4, panel labels consistent with the Phase-5
feature bank). Deletion faithfulness for **R1p and ERM only** (K3) — the metric uses no
labels, so it stands independent of the Phase-5 licence. Descriptive: ERM CAM energy in
the frozen predicted mask (mask–CAM agreement), empty-mask fallback rate on the panel.

In [ ]:
t_t = time.time()
man5 = pd.read_csv(ART["p5_manifest"])
assert len(man5) == 839, f"Phase-5 manifest drift: {len(man5)} (licensed 839)"
prev5 = float(man5["label"].mean())
assert abs(prev5 - 0.6221692491060786) < 1e-4, f"Phase-5 manifest prevalence drift: {prev5:.4f}"
BK = np.load(ART["p5_bank"], allow_pickle=True)
bk_lab = {}
for s_, l_ in zip(BK["ids"].tolist(), BK["labels"].astype(int).tolist()):
    bk_lab.setdefault(str(s_), int(l_))
rng_thy = np.random.RandomState(SEED)
sel = rng_thy.permutation(len(man5))[:THY_PANEL_N]
THY = man5.iloc[sel].sort_values("case").reset_index(drop=True)
assert set(["case", "first_image", "label"]).issubset(THY.columns), THY.columns.tolist()
_ok = 0
for _, r in THY.iterrows():
    _ok += int(bk_lab.get(Path(r["first_image"]).stem, -1) == int(r["label"]))
assert _ok == len(THY), f"panel/bank label mismatch on {len(THY) - _ok} cases"
print(f"ThyUS XAI panel: {len(THY)} case-first-images | malignant {int(THY.label.sum())} "
      f"({THY.label.mean():.3f})")

NT = len(THY)
THY_MET = {"gap_del_R1p": np.zeros(NT), "gap_del_ERM": np.zeros(NT),
           "del_guided_R1p": np.zeros(NT), "del_random_R1p": np.zeros(NT),
           "del_guided_ERM": np.zeros(NT), "del_random_ERM": np.zeros(NT),
           "gc_inmask": np.zeros(NT), "mask_area": np.zeros(NT)}
THY_EMPTY = np.zeros(NT, np.int64)
rng_stream_t = np.random.RandomState(SEED + 6)
for gi in range(NT):
    r = THY.iloc[gi]
    raw01 = letterbox_image(read_gray(r["first_image"]).astype(np.float32))
    base01 = cv2.GaussianBlur(raw01, (0, 0), BLUR_SIGMA)
    with torch.no_grad():
        fx = encode(trunk_frozen, stack_raw([raw01]))
        pm1 = (torch.sigmoid(predict_logits_384(fx, PF_PB, decoder)) >= PRED_THRESH).float()
        d1 = torch.cat([masked_gap(fx[nm], pm1) for nm in FEAT_SCALES], 1).cpu().numpy()
    fix = pm1[0, 0].cpu().numpy().astype(np.uint8)
    if fix.sum() < 4:
        fix = np.ones_like(fix); THY_EMPTY[gi] = 1
    p0_r1p = float(lin_apply("R1p_lesion", d1)[0])
    p0_erm, gc_native, _ = erm_forward_cam(stack_raw([raw01]))
    cam_r1p, _ = relu_l1(cam384(ela_native({nm: fx[nm][0].cpu().numpy().astype(np.float64)
                                            for nm in FEAT_SCALES},
                                           masks_native_np({"les": fix, "img": np.ones_like(fix)}),
                                           BLOCKS_LIN["R1p_lesion"][0])))
    cam_erm, _ = relu_l1(cv2.resize(gc_native, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR))
    THY_MET["gc_inmask"][gi] = energy_in(cam_erm, fix)
    THY_MET["mask_area"][gi] = float(fix.mean())
    order_rt = rng_stream_t.permutation(raw01.size)
    for m, cam, p0 in [("R1p", cam_r1p, p0_r1p), ("ERM", cam_erm, p0_erm)]:
        og = order_pixels(cam)
        del_g = [apply_step(raw01, base01, og, k, DEL_STEPS, "delete") for k in range(1, DEL_STEPS + 1)]
        del_r = [apply_step(raw01, base01, order_rt, k, DEL_STEPS, "delete") for k in range(1, DEL_STEPS + 1)]
        cg, cr = stack_raw(del_g), stack_raw(del_r)
        if m == "R1p":
            sg, sr = probs_lin_rep("R1p_lesion", cg, fix), probs_lin_rep("R1p_lesion", cr, fix)
        else:
            sg, sr = probs_erm(cg), probs_erm(cr)
        a_g = trapz_area([p0] + list(sg), 1 / DEL_STEPS)
        a_r = trapz_area([p0] + list(sr), 1 / DEL_STEPS)
        THY_MET[f"del_guided_{m}"][gi], THY_MET[f"del_random_{m}"][gi] = a_g, a_r
        THY_MET[f"gap_del_{m}"][gi] = a_r - a_g
    if (gi + 1) % 100 == 0 or gi == NT - 1:
        print(f"  thyus panel: {gi + 1}/{NT} ({(time.time() - t_t) / 60:.1f} min)")
print(f"ThyUS panel QC: empty-mask fallback {int(THY_EMPTY.sum())}/{NT} "
      f"({100 * THY_EMPTY.mean():.1f}%; full-cohort Phase-5 rate 10.9%) | "
      f"mean mask area {THY_MET['mask_area'].mean():.4f} | "
      f"ERM CAM-in-mask (agreement) {THY_MET['gc_inmask'].mean():.4f}")
np.savez_compressed(OUTPUT_ROOT / "phase6_panel_thyus.npz",
                    cases=THY["case"].to_numpy(), labels=THY["label"].to_numpy().astype(int),
                    empty=THY_EMPTY, **{k: v for k, v in THY_MET.items()})
print(f"[saved] phase6_panel_thyus.npz | thyus time {(time.time() - t_t) / 60:.1f} min")


## 11. Stability battery (descriptive; pre-declared expectations)
Seeded 160-image probes from each panel. Perturbations: Gaussian noise
($\sigma{=}0.02$ on the raw canvas, seeded) and affine jitter (±3°, ±5 px, seeded;
CAMs inverse-warped before comparison). The full pipeline (mask included) is
recomputed per perturbation. Expectation: noise correlations high (≳0.85) for both
models; affine lower; R1p-ELA ≥ ERM-Grad-CAM (exact, mask-anchored vs gradient noise).

In [ ]:
t_s = time.time()
def affine_warp(img01, ang, shx, shy):
    M = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), ang, 1.0)
    M[0, 2] += shx; M[1, 2] += shy
    return cv2.warpAffine(img01, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0.0), M

def inv_warp_cam(cam, M):
    Minv = cv2.invertAffineTransform(M)
    return cv2.warpAffine(cam.astype(np.float32), Minv, (IMG_SIZE, IMG_SIZE),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

def cam_r1p_full(raw01):
    with torch.no_grad():
        fx = encode(trunk_frozen, stack_raw([raw01]))
        pm1 = (torch.sigmoid(predict_logits_384(fx, PF_PB, decoder)) >= PRED_THRESH).float()
    fix = pm1[0, 0].cpu().numpy().astype(np.uint8)
    if fix.sum() < 4:
        fix = np.ones_like(fix)
    n61 = {nm: fx[nm][0].cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    c, _ = relu_l1(cam384(ela_native(n61, masks_native_np({"les": fix, "img": np.ones_like(fix)}),
                                     BLOCKS_LIN["R1p_lesion"][0])))
    return c

def cam_erm_full(raw01):
    _, gcn, _ = erm_forward_cam(stack_raw([raw01]))
    c, _ = relu_l1(cv2.resize(gcn, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR))
    return c

rng_stab5 = np.random.RandomState(SEED + 7)
ST5 = rng_stab5.choice(len(TN5K_TEST), size=STAB_N, replace=False).tolist()
rng_stabt = np.random.RandomState(SEED + 8)
STT = rng_stabt.choice(len(THY), size=STAB_N, replace=False).tolist()
STAB = []
for tag, df, col in [("TN5000-test", TN5K_TEST, "image"), ("ThyUS-600", THY, "first_image")]:
    idxs = ST5 if tag == "TN5000-test" else STT
    rng_noise = np.random.RandomState(SEED + 9)
    rng_aff = np.random.RandomState(SEED + 10)
    acc = {(m, p): [] for m in ["R1p", "ERM"] for p in ["noise", "affine"]}
    for gi in idxs:
        raw01 = letterbox_image(read_gray(df.iloc[gi][col]).astype(np.float32))
        c0 = {"R1p": cam_r1p_full(raw01), "ERM": cam_erm_full(raw01)}
        noisy = np.clip(raw01 + rng_noise.normal(0, NOISE_SD, raw01.shape).astype(np.float32), 0, 1)
        ang = rng_aff.uniform(-AFF_ROT, AFF_ROT); shx = rng_aff.uniform(-AFF_SH, AFF_SH)
        shy = rng_aff.uniform(-AFF_SH, AFF_SH)
        aff, M = affine_warp(raw01, ang, shx, shy)
        for m in ["R1p", "ERM"]:
            fn = cam_r1p_full if m == "R1p" else cam_erm_full
            acc[(m, "noise")].append(spearman_safe(c0[m], fn(noisy)))
            acc[(m, "affine")].append(spearman_safe(c0[m], inv_warp_cam(fn(aff), M)))
    for (m, p), v in acc.items():
        STAB.append({"cohort": tag, "model": m, "perturbation": p,
                     "mean Spearman": round(float(np.mean(v)), 4),
                     "median": round(float(np.median(v)), 4), "n": len(v)})
    print(f"  stability {tag}: done ({(time.time() - t_s) / 60:.1f} min)")
STAB_DF = pd.DataFrame(STAB)
display(STAB_DF)
STAB_DF.to_csv(OUTPUT_ROOT / "phase6_stability.csv", index=False)
print(f"[saved] phase6_stability.csv | stability time {(time.time() - t_s) / 60:.1f} min")


## 12. Randomisation cascade (sanity gate)
Cascading reinitialisation on a seeded 160-image TN5000 probe: R1p *head* weights
($\mathcal{N}(0, \sigma_w)$, seeded) and ERM fc → fc+layer4 → fc+layer4+layer3
(`reset_parameters`). Self-gate asserted: mean |Spearman| < 0.2 at the final cascade
stage for both models. A **JA-born check** at the end verifies no original weight was
touched (checksums).

In [ ]:
t_c = time.time()
rng_cas = np.random.RandomState(SEED + 11)
CAS_IDX = rng_cas.choice(len(TN5K_TEST), size=RAND_N, replace=False).tolist()
def rho_list(cam_fn, label=""):
    out = []
    for gi in CAS_IDX:
        raw01 = letterbox_image(read_gray(TN5K_TEST.iloc[gi]["image"]).astype(np.float32))
        out.append((gi, cam_fn(raw01)))
    return out

ORIG_R1P = rho_list(cam_r1p_full)
ORIG_ERM = rho_list(cam_erm_full)
# R1p head randomisation (pure weight-space; locked descriptors)
w_r1p = np.concatenate([b[2] for b in BLOCKS_LIN["R1p_lesion"][0]])
rng_w = np.random.RandomState(SEED + 14)
w_rand = rng_w.normal(0.0, w_r1p.std(), w_r1p.shape)
BLO = BLOCKS_LIN["R1p_lesion"][0]
blocks_rand = [(r, s, w_rand[0:256] if sc == "e3" else w_rand[256:768]) for (r, sc, _), (r_, sc__, _o) in
               zip(BLO, BLO)]
blocks_rand = [("les", "e3", w_rand[:256]), ("les", "e4", w_rand[256:])]
rho_head = []
for gi, c0 in ORIG_R1P:
    raw01 = letterbox_image(read_gray(TN5K_TEST.iloc[gi]["image"]).astype(np.float32))
    with torch.no_grad():
        fx = encode(trunk_frozen, stack_raw([raw01]))
        pm1 = (torch.sigmoid(predict_logits_384(fx, PF_PB, decoder)) >= PRED_THRESH).float()
    fix = pm1[0, 0].cpu().numpy().astype(np.uint8)
    if fix.sum() < 4:
        fix = np.ones_like(fix)
    n61 = {nm: fx[nm][0].cpu().numpy().astype(np.float64) for nm in FEAT_SCALES}
    cr, _ = relu_l1(cam384(ela_native(n61, masks_native_np({"les": fix, "img": np.ones_like(fix)}),
                                      blocks_rand)))
    rho_head.append(spearman_safe(c0, cr))
# ERM cascade on a fresh copy
def build_erm_fresh():
    m = tvm.resnet34(weights=None); m.fc = nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(load_ckpt(ART["erm_ckpt"])["model"]); m = m.to(DEVICE).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m
def reinit(mod):
    for mm in mod.modules():
        if hasattr(mm, "reset_parameters"):
            mm.reset_parameters()
def cam_of(m):
    def fn(raw01):
        global erm
        bak = erm
        try:
            erm = m
            return cam_erm_full(raw01)
        finally:
            erm = bak
    return fn
erm_c = build_erm_fresh()
CASC = []
stages = [("fc", lambda: reinit(erm_c.fc)),
          ("fc+layer4", lambda: (reinit(erm_c.fc), reinit(erm_c.layer4))),
          ("fc+layer4+layer3", lambda: (reinit(erm_c.fc), reinit(erm_c.layer4), reinit(erm_c.layer3)))]
for sname, fn in stages:
    fn()
    rhos = [spearman_safe(c0, c1) for (gi, c0), (gi2, c1) in zip(ORIG_ERM, rho_list(cam_of(erm_c)))]
    CASC.append({"stage": sname, "mean |rho|": round(float(np.mean(np.abs(rhos))), 4),
                 "mean rho": round(float(np.mean(rhos)), 4), "n": len(rhos)})
    print(f"  cascade {sname}: mean|rho| {np.mean(np.abs(rhos)):.4f} ({(time.time() - t_c) / 60:.1f} min)")
rho_head_mean = float(np.mean(np.abs(rho_head)))
CASC.append({"stage": "R1p head weights", "mean |rho|": round(rho_head_mean, 4),
             "mean rho": round(float(np.mean(rho_head)), 4), "n": len(rho_head)})
CASC_DF = pd.DataFrame(CASC)
display(CASC_DF)
gate_erm_final = CASC[-2]["mean |rho|"] < RAND_GATE_RHO
gate_r1p = CASC[-1]["mean |rho|"] < RAND_GATE_RHO
print(f"cascade gates: ERM final {CASC[-2]['mean |rho|']:.4f} < {RAND_GATE_RHO}: {gate_erm_final} | "
      f"R1p head {CASC[-1]['mean |rho|']:.4f} < {RAND_GATE_RHO}: {gate_r1p}")
assert gate_erm_final and gate_r1p, "randomisation sanity gate failed - do not interpret CAMs"
# originals-untouched checksum (JA-born)
_chk = float(torch.abs(erm.fc.weight - build_erm_fresh().fc.weight).max().item()) < 1e-12
assert _chk, "original ERM weights were mutated"
CASC_DF.to_csv(OUTPUT_ROOT / "phase6_cascade.csv", index=False)
print(f"[saved] phase6_cascade.csv | cascade time {(time.time() - t_c) / 60:.1f} min")


## 13. Pre-registered family K
Image level; 5,000 paired bootstrap resamples; two-sided empirical $p$;
Bonferroni $\alpha = 0.05/4 = 0.0125$; seed 42.
K1 localisation ratio R1p − ERM (TN5000; PRIMARY) · K2 deletion gap R1p − ERM (TN5000) ·
K3 deletion gap R1p − ERM (ThyUS-600; label-free) · K4 deletion gap JOINT-g − JOINT-img
(TN5000; G3 mirror). K1 excludes images with degenerate/empty boxes (reported).

In [ ]:
tests = []
boxok = MET["boxfrac"] > 1.5e-9      # degenerate-box guard (area strictly > 1.5e-9 of canvas)
d_k1 = (MET["r1p_ratio"] - MET["gc_ratio"])[boxok]
tests.append(("K1", "localisation ratio", "R1p-ELA", "ERM-GradCAM", d_k1,
              "PRIMARY plausibility (TN5000 expert boxes)"))
d_k2 = CUR["R1p"]["gap_del"] - CUR["ERM"]["gap_del"]
tests.append(("K2", "deletion gap", "R1p", "ERM", d_k2, "faithfulness (TN5000)"))
d_k3 = THY_MET["gap_del_R1p"] - THY_MET["gap_del_ERM"]
tests.append(("K3", "deletion gap", "R1p", "ERM", d_k3, "faithfulness (ThyUS-600; label-free)"))
d_k4 = CUR["JOINT-g"]["gap_del"] - CUR["JOINT-img"]["gap_del"]
tests.append(("K4", "deletion gap", "JOINT-g", "JOINT-img", d_k4,
              "grounding vs multi-tasking (G3 mirror)"))
rows = []
for tid, kind, a, b, d, note in tests:
    m_, lo, hi, p, n = paired_mean_boot(d)
    rows.append({"test": tid, "metric": kind, "A": a, "B": b, "delta(A-B)": round(float(m_), 4),
                 "CI95": f"({lo:.4f}..{hi:.4f})", "p_two_sided": round(float(p), 5),
                 "sig_bonf": "YES" if p < ALPHA_K else "no", "n_boot": int(n),
                 "n_items": int(np.asarray(d).size), "note": note})
FAMK = pd.DataFrame(rows)
print(f"Bonferroni alpha = {ALPHA_K:.4f} | K1 images used: {int(boxok.sum())}/{len(boxok)}")
display(FAMK)
FAMK.to_csv(OUTPUT_ROOT / "phase6_family_k.csv", index=False)
print("[saved] phase6_family_k.csv")


## 14. Run summary + artifacts
**Send back (one by one):** (1) §4 gates print (archive ERM/R1p), (2) §7 live gates +
ELA exactness + panel QC, (3) §9 faithfulness table, (4) §10 ThyUS QC lines,
(5) §11 stability table, (6) §12 cascade table + gate lines, (7) §13 family-K table,
(8) `run_summary.json` verbatim.

In [ ]:
run = {"phase": "6 - lesion-grounded XAI battery (pre-registered family K)",
       "seed": SEED, "device": str(DEVICE),
       "protocol": {"N_BOOT_K": N_BOOT_K, "ALPHA_K": ALPHA_K, "THY_PANEL_N": THY_PANEL_N,
                    "DEL_STEPS": DEL_STEPS, "BLUR_SIGMA": BLUR_SIGMA,
                    "ELA_EXACT_TOL": ELA_EXACT_TOL, "ELA_PROBE_N": ELA_PROBE_N,
                    "STAB_N": STAB_N, "RAND_N": RAND_N, "RAND_GATE_RHO": RAND_GATE_RHO,
                    "SCORECAM_TOP": SCORECAM_TOP, "SCORECAM_N": SCORECAM_N,
                    "no_refit": True, "backward_only_for_gradcam": True},
       "gates": {"erm_archive": float(erm_ref), "r1p_archive": float(r1p_ref),
                 "erm_live": float(erm_live), "r1p_live": float(r1p_live),
                 "joint_lineage_max_dp": {"g_archive_policy": lin_g, "i_archive_policy": lin_i,
                          "g_fp32_panel_note": lin_g_fp32, "i_fp32_panel_note": lin_i_fp32,
                          "tol": 1e-4, "policy": "archive-matched autocast"},
                 "ela_exactness_max_dev": {k: float(v) for k, v in ELA_DEV.items()},
                 "cascade_gate_rho": RAND_GATE_RHO,
                 "cascade_pass": bool(gate_erm_final and gate_r1p)},
       "cohorts": {"tn5000_test": {"n": int(N5), "malignant": int(Y.sum()),
                                    "empty_mask_fallbacks": int(EMPTY_FLAG.sum()),
                                    "mean_mask_area": float(MASK_AREA.mean()),
                                    "k1_boxes_used": int(boxok.sum())},
                   "thyus_panel": {"n": int(NT), "malignant": int(THY.label.sum()),
                                   "empty_mask_fallbacks": int(THY_EMPTY.sum()),
                                   "mean_mask_area": float(THY_MET["mask_area"].mean()),
                                   "erm_cam_in_mask": float(THY_MET["gc_inmask"].mean())}},
       "plausibility_tn5000": {k: {"mean": float(np.nanmean(MET[k])),
                                   "median": float(np.nanmedian(MET[k]))}
                               for k in ["r1p_ratio", "gc_ratio", "eig_ratio", "sc_ratio",
                                          "r1p_hit", "gc_hit", "eig_hit", "sc_hit",
                                          "r1p_inbox", "gc_inbox", "gc_inmask",
                                          "gc_inring", "r1p_ent", "gc_ent"]
                               if not np.isnan(MET[k]).all()},
       "faithfulness_tn5000": SUM,
       "stability": STAB,
       "cascade": CASC,
       "familyK": FAMK.to_dict(orient="records")}
json.dump(run, open(OUTPUT_ROOT / "run_summary.json", "w"), indent=2,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size / 1024:9.1f} KB")
